# Alethe — Quickstart

Two-minute tour of the `alethe` library API.

**Install:** `pip install -e ".[all]"` from the repo root, then run this notebook.

Four sections:
1. Semiring algebra — `BEYOND` taint through a join, no special-case code
2. Delta Lake watermark — real table, real VACUUM, real time-travel reads
3. Apache Iceberg watermark — same API, different metadata model
4. PIT achievability report — weakest-link composition across upstreams

In [ ]:
import alethe
print(f"alethe {alethe.__version__}")

---
## 1. Semiring algebra

`K = {ABSENT, BEYOND, OBSERVED}` with ⊕ = max, ⊗ = min.
All 126 semiring laws verified exhaustively over the 3-element carrier.

In [ ]:
from alethe import K, verify_semiring_laws

failures = verify_semiring_laws()
print(f"Semiring law check: {'ALL PASS' if not failures else failures}")

# Truth tables
print("\n⊕ (union / alternatives) — OBSERVED absorbs:")
for a in K:
    for b in K:
        from alethe._semiring import k_add
        print(f"  {a.name:8} ⊕ {b.name:8} = {k_add(a, b).name}")

print("\n⊗ (join / conjunction) — ABSENT annihilates:")
for a in K:
    for b in K:
        from alethe._semiring import k_mul
        print(f"  {a.name:8} ⊗ {b.name:8} = {k_mul(a, b).name}")

### BEYOND taint flows through a join

Two tables with mismatched retention. The join at a time before `orders`'
watermark should surface `BEYOND` — not an empty answer, not an error.

In [ ]:
from alethe import TemporalTable, KRelation, split_result

# customers: long retention (all history from t=0)
customers = TemporalTable(
    name="customers",
    columns=("customer_id", "segment"),
    key_columns=("customer_id",),
    retention_watermark=0,
)
customers.insert_version(("C1", "smb"), 0, 50)
customers.insert_version(("C1", "enterprise"), 50)
customers.insert_version(("C2", "smb"), 10)

# orders: vacuumed at t=40 — history before t=40 is destroyed
orders = TemporalTable(
    name="orders",
    columns=("order_id", "customer_id", "amount"),
    key_columns=("order_id",),
    retention_watermark=40,
)
orders.insert_version(("O1", "C1", 100), 20)  # pre-watermark!
orders.insert_version(("O2", "C2", 250), 45)
orders.insert_version(("O3", "C1", 900), 60)

# Safe query (t=60, both tables intact)
c60 = customers.as_of(60)
o60 = orders.as_of(60)
result60 = split_result(o60.join(c60, on=["customer_id"]))
print(f"AS OF t=60 → refused={result60.refused}, answers={len(result60.answers.data)}")

# Crossing the boundary (t=30, orders vacuumed)
c30 = customers.as_of(30)
o30 = orders.as_of(30)
result30 = split_result(o30.join(c30, on=["customer_id"]))
print(f"AS OF t=30 → refused={result30.refused}, refusals={len(result30.refusals.data)}")
print("  BEYOND taint propagated from orders through ⊗ into the join — zero special-case code.")

---
## 2. Delta Lake watermark

Real `DeltaTable`, real `VACUUM`, real time-travel reads.
The oracle proves that `_delta_log` lists versions that are physically unreadable.

In [ ]:
from pathlib import Path
import pandas as pd
from deltalake import DeltaTable, write_deltalake

TABLE = Path(".") / "lakehouse" / "qs_orders"
TABLE.parent.mkdir(parents=True, exist_ok=True)

# Write 6 versions (overwrite each time so old files become unreferenced)
for v in range(6):
    df = pd.DataFrame({
        "order_id": [f"O{v}{i}" for i in range(4)],
        "amount":   [100.0 * (v + 1) + i for i in range(4)],
    })
    write_deltalake(TABLE, df, mode="overwrite")

print(f"Wrote 6 versions → {TABLE}")

# Vacuum with zero retention (the footgun OWS makes honest)
dt = DeltaTable(str(TABLE))
removed = dt.vacuum(retention_hours=0, enforce_retention_duration=False, dry_run=False)
print(f"VACUUM removed {len(removed)} parquet file(s)")

In [ ]:
import alethe

wm = alethe.watermark(TABLE)

print(f"chain:                {wm.chain}")
print(f"boundary:             version {wm.boundary['version']}")
print(f"boundary_dt:          {wm.boundary_dt}")
print(f"evidence_grade:       {wm.evidence_grade}")
print(f"empirically_validated:{wm.empirically_validated}")
print(f"corroborated:         {wm.proof['corroborated']}")

In [ ]:
from datetime import datetime, timezone

# Verdict for a query starting well after the boundary
v_safe = alethe.verdict(wm, since=wm.boundary_dt)
print(f"Query at boundary → {v_safe}")

# Verdict for a query starting before the boundary
from datetime import timedelta
v_stale = alethe.verdict(wm, since=wm.boundary_dt - timedelta(days=30))
print(f"Query 30 days before boundary → {v_stale}")

In [ ]:
# Persist to manifest
entry = alethe.record(wm, "qs_manifest.jsonl")
print(f"Manifest entry: seq={entry['seq']}  hash={entry['hash']}")

# Verify chain integrity
from alethe import Manifest
m = Manifest("qs_manifest.jsonl")
print(f"Chain: {m}")

---
## 3. Apache Iceberg watermark

Same `alethe.watermark()` call, different adapter.
Iceberg uses snapshot IDs and sequence numbers instead of version integers.

In [ ]:
import shutil
import pyarrow as pa
from pyiceberg.catalog.sql import SqlCatalog

WAREHOUSE = Path(".") / "iceberg_warehouse_qs"
if WAREHOUSE.exists():
    shutil.rmtree(WAREHOUSE)
WAREHOUSE.mkdir()

catalog = SqlCatalog(
    "local",
    uri=f"sqlite:///{WAREHOUSE}/catalog.db",
    warehouse=f"file://{WAREHOUSE.resolve()}",
)
catalog.create_namespace("demo")
schema = pa.schema([("order_id", pa.string()), ("amount", pa.float64())])
tbl = catalog.create_table("demo.orders", schema=schema)

# 5 overwrites → 5 snapshots, old data files become unreferenced
for s in range(5):
    batch = pa.table({
        "order_id": [f"O{s}{i}" for i in range(3)],
        "amount":   [float(100 * (s + 1) + i) for i in range(3)],
    })
    tbl.overwrite(batch)

print(f"Built Iceberg table with {len(tbl.metadata.snapshots)} snapshots")

In [ ]:
# Destroy old data files out-of-band (simulates aggressive lifecycle policy)
snapshots = sorted(tbl.metadata.snapshots, key=lambda s: s.sequence_number)
keep_files: set[str] = set()
for snap in snapshots[-2:]:
    for task in tbl.scan(snapshot_id=snap.snapshot_id).plan_files():
        keep_files.add(task.file.file_path)

removed = 0
for snap in snapshots[:-2]:
    try:
        for task in tbl.scan(snapshot_id=snap.snapshot_id).plan_files():
            p = Path(task.file.file_path.replace("file://", ""))
            if task.file.file_path not in keep_files and p.exists():
                p.unlink()
                removed += 1
    except Exception:
        pass

print(f"Deleted {removed} old data file(s) — metadata still lists all {len(snapshots)} snapshots")

In [ ]:
wm_ice = alethe.watermark("demo.orders", adapter="iceberg", catalog=catalog)

print(f"chain:                {wm_ice.chain}")
print(f"boundary snapshot_id: {wm_ice.boundary['snapshot_id'][:16]}…")
print(f"boundary sequence:    {wm_ice.boundary['sequence_number']}")
print(f"boundary_dt:          {wm_ice.boundary_dt}")
print(f"evidence_grade:       {wm_ice.evidence_grade}")
print(f"empirically_validated:{wm_ice.empirically_validated}")
print(f"snapshots listed:     {wm_ice.proof['snapshots_listed']}")
print(f"snapshots readable:   {wm_ice.proof['snapshots_readable']}")
print(f"readable islands:     {len(wm_ice.readable_islands)} below boundary")

In [ ]:
# Append Iceberg watermark to the same manifest
entry_ice = alethe.record(wm_ice, "qs_manifest.jsonl")
print(f"Manifest entry: seq={entry_ice['seq']}  hash={entry_ice['hash']}")

m = Manifest("qs_manifest.jsonl")
print(f"Final manifest: {m}  ← two formats, one ledger")

---
## 4. PIT achievability report

`alethe.pit_report(name, upstreams)` applies weakest-link composition
(spec §4) across any number of upstream watermarks and divides the
timeline into three zones:

| Zone | Condition | Meaning |
|---|---|---|
| **CERTAIN** | `since ≥ effective_boundary` | All upstream history retained — exact answer |
| **BOUNDED** | `earliest_dt ≤ since < effective_boundary` | Data exists but some is vacuumed — monotone aggregates valid as lower bounds |
| **UNACHIEVABLE** | `since < earliest_dt` | At least one upstream didn't exist yet — population itself is unknowable |

The effective boundary is the **most restrictive** (latest) boundary across all
upstreams. Effective grade is the **weakest** grade on the path.

In [ ]:
# wm (Delta) and wm_ice (Iceberg) are computed in sections 2 and 3.
# Build a report for a hypothetical downstream model that joins both.
report = alethe.pit_report("reporting.revenue_summary", [wm, wm_ice])
print(report)

In [ ]:
from datetime import timedelta

# Check specific query times against the report
test_times = [
    ("now",              wm.boundary_dt + timedelta(days=1)),
    ("at boundary",      report.effective_boundary),
    ("inside BOUNDED",   report.effective_boundary - timedelta(days=1)),
    ("at earliest",      report.earliest_dt),
    ("before any data",  report.earliest_dt - timedelta(days=30)),
]

print(f"{'Query time':<20}  Zone")
print("─" * 45)
for label, t in test_times:
    zone = report.query(t)
    note = f"  (limiting: {zone.limiting_chains})" if zone.limiting_chains else ""
    print(f"{label:<20}  {zone.status.value}{note}")